# 02 · Limpieza de clima

**Entrada:** seis Excel originales acumulativos de `Datos/Clima/`.
**Salida única:** `clima_limpio.parquet`, una fila por finca/instante de medición.

Se leen las hojas de datos, no sus tablas resumen. Las mediciones repetidas entre libros se contabilizan una vez. Cuando distintas ediciones discrepan se conserva la primera edición y se marca conflicto; no se declara automáticamente válida.

## 1. Imports y configuración

In [1]:
from pathlib import Path
import sys
import re
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "Datos").is_dir() and (p / "Notebooks").is_dir())
from datetime import date
import unicodedata
from pathlib import PurePosixPath
from zipfile import ZipFile
from lxml import etree

def texto(valor):
    if pd.isna(valor):
        return ""
    s = unicodedata.normalize("NFKD", str(valor).replace("\xa0", " "))
    return re.sub(r"\s+", " ", "".join(c for c in s if not unicodedata.combining(c))).strip().upper()

def normalizar(serie):
    return (serie.astype("string").str.normalize("NFKD")
            .str.replace("[\u0300-\u036f]", "", regex=True)
            .str.replace(r"\s+", " ", regex=True).str.strip().str.upper()
            .replace({"": pd.NA, "NAN": pd.NA, "NONE": pd.NA, "<NA>": pd.NA}))

def finca(serie):
    return normalizar(serie).replace({"GF": "GAITANA", "LA GAITANA": "GAITANA", "¡": "GAITANA",
                                     "AR": "ARABELLA", "LA CABANA": "CABANA", "FINCA LA CABANA": "CABANA"})

def fechas(serie):
    """Conservar calendario de Excel: fechas ya leídas o seriales 1900, no nanosegundos 1970."""
    salida = pd.to_datetime(serie, errors="coerce", format="mixed")
    numeros = pd.to_numeric(serie.astype("string"), errors="coerce")
    serial = numeros.between(20000, 80000)
    salida.loc[serial] = pd.to_datetime(numeros[serial], unit="D", origin="1899-12-30")
    return salida

def semana_operativa(anio, semana):
    """Lunes representativo. Toda etiqueta 53 se convierte en 01 del año siguiente."""
    y = pd.to_numeric(anio, errors="coerce").astype("Int64")
    w = pd.to_numeric(semana, errors="coerce").astype("Int64")
    y = y + w.eq(53).fillna(False).astype(int)
    w = w.mask(w.eq(53), 1)
    etiquetas = y.astype("string") + "-" + w.astype("string")
    conversion = {}
    for etiqueta in etiquetas.dropna().unique():
        try:
            yy, ww = map(int, etiqueta.split("-"))
            conversion[etiqueta] = pd.Timestamp(date.fromisocalendar(yy, ww, 1)) if 1 <= ww <= 52 else pd.NaT
        except (ValueError, OverflowError):
            conversion[etiqueta] = pd.NaT
    return pd.to_datetime(etiquetas.map(conversion))

SALIDA = ROOT / "Datos_analiticos_proyecto"
SALIDA.mkdir(exist_ok=True)
pd.set_option("display.max_columns", 35)
print("Fuentes originales:", ROOT / "Datos")
print("Salida oficial:", SALIDA)

ENTRADA = ROOT / "Datos/Clima"
ARCHIVO_SALIDA = SALIDA / "clima_limpio.parquet"
archivos = sorted(ENTRADA.glob("*.xlsx"),key=lambda p:(int(re.search(r"20\d{2}",p.name).group()),p.name))
display(pd.DataFrame({"archivo":[p.name for p in archivos]}))


Fuentes originales: C:\Materias y Cosas Técnicas Uniandes\Proyecto de grado\Proyecto de grado 2\Datos
Salida oficial: C:\Materias y Cosas Técnicas Uniandes\Proyecto de grado\Proyecto de grado 2\Datos_analiticos_proyecto


,archivo
0,NUEVO Informe variables climáticas Gaitana-Ara...
1,NUEVO Informe variables climáticas Gaitana-Ara...
2,Nuevo Informe Clima 2023...xlsx
3,Informe Clima 2024.xlsx
4,Informe Clima 2025_2.xlsx
5,Informe Clima 2026.xlsx


## 2. Estructura de las hojas originales

2021/2022 separan Gaitana y Arabella; los libros posteriores usan BD con SEDE. La finca viene de la hoja o SEDE, no se infiere desde el nombre de un código de estación.

In [2]:
x = pd.ExcelFile(archivos[0],engine="calamine")
print(x.sheet_names)
display(pd.read_excel(x,sheet_name="Base de datos Gaitana",nrows=5))
x.close()


['Tablas resumenGaitana', 'Base de datos Gaitana', 'Base de datos2', 'Estación Gaitana', 'Estación Arabella', 'TablasArabella1', 'Tablas Arabella2', 'Base de datos Arabella', 'BD2Arabella']


,Año,Mes,Semana,DIA,Date,Time,Tem Out,Hi Temp,Low Temp,Out Hum,Dew Pt.,Wind Speed,Wind Run,Hi Speed,Wind Chill,Heat Index,THW Index,...,Hi UV,Het D-D,Cool D-D,In Temp,In Hum,In Dew,In Heat,In EMC,In AirDensity,ET,EVT,Precipitación,Promedio EVT,Promedio precipitación,Unnamed: 39,0,0.1
0,2017,9,38,19,2017-09-19,11:00:00,19.5,19.8,19.5,60,11.5,4.8,2.41,19.3,19.5,19.2,19.2,...,10.7,0,0.024,23.9,46,11.6,23.8,8.55,1.1640,0.43,2.71,0.0,2.682172,23.370263,49.0,NaN,NaN
1,2017,9,38,19,2017-09-19,11:30:00,19.2,19.7,19.2,62,11.8,6.4,3.22,17.7,19.2,19.0,19.0,...,10.8,0,0.019,23.7,49,12.4,23.7,9.05,1.1635,0.00,NaN,NaN,2.682172,23.370263,NaN,NaN,NaN
2,2017,9,38,19,2017-09-19,12:00:00,19.8,19.8,19.6,63,12.5,6.4,3.22,19.3,19.8,19.7,19.7,...,9.6,0,0.030,23.3,50,12.3,23.2,9.25,1.1643,0.43,NaN,NaN,2.682172,23.370263,NaN,NaN,NaN
3,2017,9,38,19,2017-09-19,12:30:00,19.8,19.9,19.6,64,12.8,8.0,4.02,20.9,19.8,19.8,19.8,...,10.7,0,0.031,23.1,55,13.5,23.1,10.08,1.1636,0.00,NaN,NaN,2.682172,23.370263,NaN,NaN,NaN
4,2017,9,38,19,2017-09-19,13:00:00,19.8,19.9,19.3,66,13.2,6.4,3.22,17.7,19.8,19.9,19.9,...,8.3,0,0.030,22.7,53,12.6,22.6,9.73,1.1653,0.41,NaN,NaN,2.682172,23.370263,NaN,NaN,NaN


## 3. Carga, tipos, selección y procedencia

Se conserva fecha/hora original y mediciones desde 2021. La temperatura/humedad/radiación se validan más abajo. Lluvia y evapotranspiración se preservan, pero su unidad/periodo de acumulación deben confirmarse antes de modelarlas.

In [3]:
alias = {"SEDE":"finca","DATE":"fecha","TIME":"hora","TEM OUT":"temperatura","TEMP OUT":"temperatura",
         "HI TEMP":"temperatura_max","LOW TEMP":"temperatura_min","OUT HUM":"humedad",
         "RAIN":"lluvia","SOLAR RAD.":"radiacion","ET":"evapotranspiracion","CODIGO":"codigo_fuente"}
partes, controles = [], []
for archivo in archivos:
    libro = pd.ExcelFile(archivo,engine="calamine")
    hojas = ["BD"] if "BD" in libro.sheet_names else ["Base de datos Gaitana","Base de datos Arabella"]
    for hoja in hojas:
        df = pd.read_excel(libro,sheet_name=hoja,usecols=lambda c:texto(c) in alias)
        df.columns = [alias[texto(c)] for c in df.columns]
        df["fila_excel"] = np.arange(2,len(df)+2)
        if "finca" not in df:
            df["finca"] = "GAITANA" if "Gaitana" in hoja else "ARABELLA"
        df["finca"] = finca(df.finca)
        df["fecha"] = fechas(df.fecha).dt.normalize()
        df["hora_original"] = df.hora.astype("string")
        # Excel/calamine devuelve datetime.time; su representación HH:MM:SS es inequívoca.
        df["hora_con_zona"] = df.hora_original.str.contains(r"[+-]\d{2}:\d{2}$",regex=True,na=False)
        # Usamos fecha/hora de pared reportada, sin inferir conversión UTC/local.
        hora_texto = df.hora_original.str.replace(r"[+-]\d{2}:\d{2}$","",regex=True)
        hora = pd.to_timedelta(hora_texto,errors="coerce")
        hora_num = pd.to_numeric(df.hora.astype("string"),errors="coerce")
        hora = hora.where(hora.notna(),pd.to_timedelta(hora_num,unit="D"))
        df["instante"] = df.fecha + hora
        n_sin_fecha = int(df.fecha.isna().sum())
        df = df.loc[df.fecha.ge("2021-01-01")].copy()
        for col in ["temperatura","temperatura_max","temperatura_min","humedad","lluvia","radiacion","evapotranspiracion"]:
            df[col] = pd.to_numeric(df[col],errors="coerce")
        df["archivo_origen"] = str(archivo.relative_to(ROOT)).replace("\\","/")
        df["hoja_origen"] = hoja
        df["edicion_anio"] = int(re.search(r"20\d{2}",archivo.name).group())
        partes.append(df.drop(columns=["hora"]))
        controles.append({"archivo":archivo.name,"hoja":hoja,"filas_desde_2021":len(df),"sin_fecha_original":n_sin_fecha})
        print(archivo.name,hoja,len(df),flush=True)
    libro.close()
clima = pd.concat(partes,ignore_index=True)
del partes, df
display(pd.DataFrame(controles))


NUEVO Informe variables climáticas Gaitana-Arabella-2021.xlsx Base de datos Gaitana 15089


NUEVO Informe variables climáticas Gaitana-Arabella-2021.xlsx Base de datos Arabella 18374


NUEVO Informe variables climáticas Gaitana-Arabella-2022..xlsx Base de datos Gaitana 31863


NUEVO Informe variables climáticas Gaitana-Arabella-2022..xlsx Base de datos Arabella 37864


Nuevo Informe Clima 2023...xlsx BD 98689


Informe Clima 2024.xlsx BD 132955


Informe Clima 2025_2.xlsx BD 168434


Informe Clima 2026.xlsx BD 223016


,archivo,hoja,filas_desde_2021,sin_fecha_original
0,NUEVO Informe variables climáticas Gaitana-Ara...,Base de datos Gaitana,15089,0
1,NUEVO Informe variables climáticas Gaitana-Ara...,Base de datos Arabella,18374,0
2,NUEVO Informe variables climáticas Gaitana-Ara...,Base de datos Gaitana,31863,0
3,NUEVO Informe variables climáticas Gaitana-Ara...,Base de datos Arabella,37864,0
4,Nuevo Informe Clima 2023...xlsx,BD,98689,0
5,Informe Clima 2024.xlsx,BD,132955,0
6,Informe Clima 2025_2.xlsx,BD,168434,0
7,Informe Clima 2026.xlsx,BD,223016,0


## 4. No sumar libros acumulativos

Llave natural provisional: finca + instante. El campo CODIGO no se ha validado como identificador físico de estación. Si hay dos mediciones diferentes para la misma llave se marca conflicto y se excluye esa medición de promedios analíticos; no se promedia la discrepancia como si fueran dos estaciones.

In [4]:
llave = ["finca","instante"]
variables = ["temperatura","temperatura_max","temperatura_min","humedad","lluvia","radiacion","evapotranspiracion"]
clima["llave_medicion_valida"] = clima[llave].notna().all(axis=1)
sin_instante = clima.loc[~clima.llave_medicion_valida].copy()
display(sin_instante[["archivo_origen","finca","fecha","hora_original"]])
clima = clima.loc[clima.llave_medicion_valida].copy()
n_antes = len(clima)
conflictos = clima.groupby(llave,dropna=False)[variables].nunique().gt(1).any(axis=1).rename("conflicto_entre_ediciones")
repeticiones = clima.groupby(llave).size().rename("registros_fuente")
clima = clima.sort_values(["edicion_anio","archivo_origen","fila_excel"]).drop_duplicates(llave,keep="first")
clima = clima.merge(conflictos,on=llave,validate="one_to_one").merge(repeticiones,on=llave,validate="one_to_one")
clima["medicion_valida"] = (clima.temperatura.between(-5,50) & clima.humedad.between(0,100)
                            & clima.radiacion.ge(0) & ~clima.conflicto_entre_ediciones)
sin_instante["medicion_valida"] = False
sin_instante["conflicto_entre_ediciones"] = False
sin_instante["registros_fuente"] = 1
clima = pd.concat([clima,sin_instante],ignore_index=True)
iso = clima.fecha.dt.isocalendar()
clima["semana_inicio"] = semana_operativa(iso.year,iso.week)
print("Antes:",n_antes,"Mediciones únicas:",len(clima),"Conflictos:",int(clima.conflicto_entre_ediciones.sum()))
display(clima.groupby("finca").agg(desde=("fecha","min"),hasta=("fecha","max"),mediciones=("instante","size"),validas=("medicion_valida","sum")))
display(clima[variables].isna().sum().to_frame("faltantes"))
assert not clima.loc[clima.llave_medicion_valida].duplicated(llave).any()


,archivo_origen,finca,fecha,hora_original
87600,Datos/Clima/NUEVO Informe variables climáticas...,ARABELLA,2022-12-03,1900-01-03 00:00:00
159615,Datos/Clima/Nuevo Informe Clima 2023...xlsx,ARABELLA,2022-03-12,1900-01-03 00:00:00
224115,Datos/Clima/Informe Clima 2024.xlsx,ARABELLA,2022-03-12,1900-01-03 00:00:00
357070,Datos/Clima/Informe Clima 2025_2.xlsx,ARABELLA,2022-03-12,1900-01-03 00:00:00
525504,Datos/Clima/Informe Clima 2026.xlsx,ARABELLA,2022-03-12,1900-01-03 00:00:00


Antes: 726279 Mediciones únicas: 187134 Conflictos: 7365


,desde,hasta,mediciones,validas
finca,,,,
ARABELLA,2021-01-01,2026-08-11,96373,89238
GAITANA,2021-01-01,2026-07-28,90761,88919


,faltantes
temperatura,1599
temperatura_max,1603
temperatura_min,1603
humedad,1601
lluvia,5
radiacion,1544
evapotranspiracion,2609


## 5. Exportación y resumen

Las mediciones con conflicto se conservan señaladas en la misma base limpia. El notebook 06 agregará sólo mediciones admisibles; no se exporta aquí otra base semanal.

In [5]:
clima.to_parquet(ARCHIVO_SALIDA,index=False)
print("EXPORTADO:",ARCHIVO_SALIDA,clima.shape)
display(clima.head())


EXPORTADO: C:\Materias y Cosas Técnicas Uniandes\Proyecto de grado\Proyecto de grado 2\Datos_analiticos_proyecto\clima_limpio.parquet (187134, 22)


,fecha,temperatura,temperatura_max,temperatura_min,humedad,lluvia,radiacion,evapotranspiracion,fila_excel,finca,hora_original,hora_con_zona,instante,archivo_origen,hoja_origen,edicion_anio,codigo_fuente,llave_medicion_valida,conflicto_entre_ediciones,registros_fuente,medicion_valida,semana_inicio
0,2021-01-01,NaN,NaN,NaN,NaN,0.0,NaN,0.0,52564,GAITANA,00:04:00,False,2021-01-01 00:04:00,Datos/Clima/NUEVO Informe variables climáticas...,Base de datos Gaitana,2021,NaN,True,False,2,False,2021-01-04
1,2021-01-01,12.1,12.1,11.9,92.0,0.0,0.0,0.0,52565,GAITANA,00:30:00,False,2021-01-01 00:30:00,Datos/Clima/NUEVO Informe variables climáticas...,Base de datos Gaitana,2021,NaN,True,False,6,True,2021-01-04
2,2021-01-01,11.9,11.9,11.9,92.0,0.0,0.0,0.0,52566,GAITANA,02:00:00,False,2021-01-01 02:00:00,Datos/Clima/NUEVO Informe variables climáticas...,Base de datos Gaitana,2021,NaN,True,False,6,True,2021-01-04
3,2021-01-01,12.1,12.1,11.9,93.0,0.0,0.0,0.0,52567,GAITANA,02:30:00,False,2021-01-01 02:30:00,Datos/Clima/NUEVO Informe variables climáticas...,Base de datos Gaitana,2021,NaN,True,False,6,True,2021-01-04
4,2021-01-01,12.0,12.1,12.0,92.0,0.0,0.0,0.0,52568,GAITANA,03:00:00,False,2021-01-01 03:00:00,Datos/Clima/NUEVO Informe variables climáticas...,Base de datos Gaitana,2021,NaN,True,False,6,True,2021-01-04
